In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import scipy
import scipy.sparse


In [2]:
BM_sample = sc.read_h5ad('/lotterlab/datasets/VCC/DATASETS/SCFM_datasets/EvalDatasets/cell_identity/Bone_Marrow_RNA_ATAC_Neurips_2021/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad')

First, we take a bone marrow dataset of 42,492 mononuclear cells from 12 healthy human donors collected across three separate experiment sites 53. We take the data from the first site as the reference split and predict the annotations for the data from the second site, simulating a real scenario where query and reference datasets are sampled and sequenced separately in different conditions. The dataset comprises 22 cell types with an uneven distribution across the different classes (Fig. 2d) for both reference and query datasets. According to the original study, the cells are annotated using the joint transcriptome and chromatin accessibility profiles together as the data is part of a multiome assay, however here we only exploit the gene expression modality for annotation.

In [3]:
for ob in BM_sample.obs:
    print(BM_sample.obs[ob].value_counts())
    print('--------------------------------')

GEX_pct_counts_mt
0.000000    12514
2.272727       32
1.315789       29
1.960784       29
1.136364       27
            ...  
0.368596        1
1.454716        1
1.167582        1
1.910988        1
7.535642        1
Name: count, Length: 31382, dtype: int64
--------------------------------
GEX_n_counts
641.0     50
808.0     44
711.0     44
1815.0    43
787.0     41
          ..
5789.0     1
3668.0     1
4909.0     1
3780.0     1
4955.0     1
Name: count, Length: 7055, dtype: int64
--------------------------------
GEX_n_genes
1144    71
1103    68
1219    67
1194    67
499     67
        ..
4250     1
4002     1
5177     1
5002     1
3153     1
Name: count, Length: 3741, dtype: int64
--------------------------------
GEX_size_factors
0.453484    1
0.845258    1
0.477193    1
1.141015    1
0.579384    1
           ..
0.440912    1
1.075236    1
0.798237    1
0.826250    1
0.498140    1
Name: count, Length: 69249, dtype: int64
--------------------------------
GEX_phase
G2M    48352
S      

In [4]:
for va in BM_sample.var:
    print(BM_sample.var[va].value_counts())
    print('--------------------------------')


feature_types
ATAC    116490
GEX      13431
Name: count, dtype: int64
--------------------------------
gene_id
ENSG00000000419    1
ENSG00000173200    1
ENSG00000173457    1
ENSG00000173465    1
ENSG00000173473    1
                  ..
ENSG00000130734    1
ENSG00000130741    1
ENSG00000130748    1
ENSG00000130749    1
ENSG00000288380    1
Name: count, Length: 13431, dtype: int64
--------------------------------


In [5]:
for key in BM_sample.uns:
    print(f"\n{key}:")
    print(f"  Type: {type(BM_sample.uns[key])}")
    if isinstance(BM_sample.uns[key], dict):
        print(f"  Keys: {list(BM_sample.uns[key].keys())}")
    elif isinstance(BM_sample.uns[key], np.ndarray):
        print(f"  Shape: {BM_sample.uns[key].shape}")
        print(f"  Dtype: {BM_sample.uns[key].dtype}")
    else:
        print(f"  Value: {BM_sample.uns[key]}")



ATAC_gene_activity_var_names:
  Type: <class 'numpy.ndarray'>
  Shape: (19039,)
  Dtype: object

dataset_id:
  Type: <class 'str'>
  Value: openproblems_bmmc_multiome

genome:
  Type: <class 'str'>
  Value: GRCh38

organism:
  Type: <class 'str'>
  Value: human


In [6]:

# 2. Inspect obsm
for key in BM_sample.obsm:
    print(f"\n{key}:")
    print(f"  Shape: {BM_sample.obsm[key].shape}")
    print(f"  Dtype: {BM_sample.obsm[key].dtype}")

# 3


ATAC_gene_activity:
  Shape: (69249, 19039)
  Dtype: float32

ATAC_lsi_full:
  Shape: (69249, 50)
  Dtype: float64

ATAC_lsi_red:
  Shape: (69249, 39)
  Dtype: float64

ATAC_umap:
  Shape: (69249, 2)
  Dtype: float64

GEX_X_pca:
  Shape: (69249, 50)
  Dtype: float32

GEX_X_umap:
  Shape: (69249, 2)
  Dtype: float32


In [ ]:
#. Check main data matrix
print(f"\nMain matrix (X):")
print(f"  Shape: {BM_sample.X.shape}")
print(f"  Type: {type(BM_sample.X)}")
print(f"  Sparsity: {(BM_sample.X == 0).sum() / BM_sample.X.size * 100:.2f}%")

# 4. Check layers
if 'counts' in BM_sample.layers:
    print(f"\nCounts layer:")
    print(f"  Shape: {BM_sample.layers['counts'].shape}")
    print(f"  Type: {type(BM_sample.layers['counts'])}")


Main matrix (X):
  Shape: (69249, 129921)
  Type: <class 'scipy.sparse._csr.csr_matrix'>


/lotterlab/users/riccardo/ML_BIO/SCFM_meta/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3699: SparseEfficiencyWarning: Comparing a sparse matrix with 0 using == is inefficient, try using != instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
